# COMP 9130 — Capstone Project: Military Camouflage Object Detection
## Notebook 03: Transfer Learning — Experiment 2 (SegFormer-B2)

**Author:** Sepehr Mansouri (Experiment 2 + Evaluation Lead)   
**Datasets:** COD10K (pretrain) → ACD1K (fine-tune)   
**Model:** SegFormer-B2 (nvidia/segformer-b2-finetuned-ade-512-512)

**Purpose:** Train SegFormer-B2 via two-stage cross-domain transfer learning.
Stage 1 pretrains on COD10K (animal camouflage) to acquire general camouflage
representations, then Stage 2 fine-tunes on ACD1K (military camouflage) with
a lower learning rate to adapt to the target domain while minimising
catastrophic forgetting. This tests whether animal-camouflage pretraining
transfers to military-specific targets.

---

### Notebook Structure

1. **Environment Setup** — Clone repo, install dependencies, configure Kaggle credentials
2. **Dataset Download** — COD10K (Kaggle), ACD1K (Kaggle)
3. **Dataset Verification** — Confirm all folder paths and split index files
4. **GPU Verification** — Confirm A100 is available
5. **Stage 1 — COD10K Pretraining** — Train on animal camouflage with differential LR
6. **Stage 1 Results** — Inspect best checkpoint
7. **Stage 2 — Hyperparameter Sweep** — Three 20-epoch runs on ACD1K across learning rates [1e-4, 1e-5, 5e-6]
8. **Sweep Results** — Compare val mIoU across runs, select best LR
9. **Stage 2 — Final Training Run** — 50-epoch run with best LR, early stopping patience=10
10. **Results & Export** — Save history.json, push to GitHub, download checkpoint

---

### Experiment Design

| Setting | Value |
|---|---|
| Model | SegFormer-B2 |
| Pretrained checkpoint | nvidia/segformer-b2-finetuned-ade-512-512 |
| **Stage 1 — COD10K Pretrain** | |
| Training data | COD10K (5,950 images) |
| Validation data | COD10K val (3,950 images) |
| Encoder LR | 1e-4 |
| Decode head LR | 1e-3 |
| **Stage 2 — ACD1K Fine-tune** | |
| Training data | ACD1K (748 images) |
| Validation data | ACD1K val (230 images) |
| Sweep LRs | 1e-4, 1e-5, 5e-6 |
| Final LR | TBD (best from sweep) |
| **Common** | |
| Input resolution | 512 × 512 |
| Batch size | 16 (A100) |
| Loss function | Binary cross-entropy + Dice |
| Early stopping patience | 10 epochs |
| Hardware | Google Colab A100 GPU |

---
## 1. Environment Setup

In [ ]:
# Cell 1 — Clone repo
!git clone https://github.com/bing-er/AI-final-project.git
%cd /content/AI-final-project

In [ ]:
# Cell 2 — Install dependencies
!pip install -q transformers albumentations kaggle

In [ ]:
# Cell 3 — Upload kaggle.json (run once per session)
import os
from google.colab import files

files.upload()  # select kaggle.json from your machine when prompted
os.makedirs('/root/.config/kaggle', exist_ok=True)
os.rename('kaggle.json', '/root/.config/kaggle/kaggle.json')
os.chmod('/root/.config/kaggle/kaggle.json', 0o600)
print('Kaggle credentials configured')

---
## 2. Dataset Download

Experiment 2 uses only **COD10K** and **ACD1K** (no CAMO needed).

In [ ]:
# Cell 4 — Download COD10K (~1.2 GB)
import os
if not os.path.exists('data/COD10K-v3/Train/Image'):
    !kaggle datasets download \
        -d ivanomelchenkoim11/cod10k-dataset \
        -p data/ --unzip
else:
    print("✅ COD10K already exists, skipping download")

In [ ]:
# Cell 5 — Download ACD1K (~350 MB)
import os
if not os.path.exists('data/dataset-splitM/Training/images'):
    !kaggle datasets download \
        -d aalihhiader/military-camouflage-soldiers-dataset-mcs1k \
        -p data/ --unzip
else:
    print("✅ ACD1K already exists, skipping download")

---
## 3. Dataset & Split Verification

In [ ]:
# Cell 6 — Verify folder structure
import os
expected = [
    'data/COD10K-v3/Train/Image',
    'data/COD10K-v3/Train/GT_Object',
    'data/COD10K-v3/Test/Image',
    'data/COD10K-v3/Test/GT_Object',
    'data/dataset-splitM/Training/images',
    'data/dataset-splitM/Training/GT',
    'data/dataset-splitM/Testing/images',
    'data/dataset-splitM/Testing/GT',
]
for p in expected:
    status = '✅' if os.path.exists(p) else '❌ NOT FOUND'
    print(f'{status} — {p}')

---
## 4. GPU Verification

In [ ]:
# Cell 7 — Verify GPU
!nvidia-smi

In [ ]:
# Cell 8 — Verify dataset loading (COD10K and ACD1K conditions)
!PYTHONPATH=/content/AI-final-project/src \
 python /content/AI-final-project/src/dataset.py \
    /content/AI-final-project/data/ \
    /content/AI-final-project/splits/

---
## 5. Stage 1 — COD10K Pretraining

Stage 1 pretrains SegFormer-B2 on COD10K (5,950 animal camouflage images)
to learn general camouflage-detection features before domain-specific
fine-tuning on military imagery.

**Differential learning rate:** The MiT-B2 encoder (initialised from ADE20K)
uses a lower LR (1e-4) to preserve pretrained spatial features, while the
randomly-initialised binary decode head uses a higher LR (1e-3) to train
faster from scratch.

First, we need to add `train_exp2.py` to the repo if it's not already there.

In [ ]:
# Cell 9 — Check if train_exp2.py exists, upload if not
import os
if not os.path.exists('src/train_exp2.py'):
    print("train_exp2.py not found in src/ — uploading...")
    from google.colab import files
    uploaded = files.upload()  # upload train_exp2.py
    os.rename('train_exp2.py', 'src/train_exp2.py')
    print("✅ train_exp2.py placed in src/")
else:
    print("✅ train_exp2.py already exists in src/")

In [ ]:
# Cell 10 — Stage 1: COD10K pretraining (differential LR)
!PYTHONPATH=/content/AI-final-project/src \
 python src/train_exp2.py \
    --stage 1 \
    --encoder_lr 1e-4 --head_lr 1e-3 \
    --epochs 50 --batch_size 16 --accum_steps 1 \
    --data_root data/ --splits_dir splits/ \
    --output_dir outputs/exp2/stage1

---
## 6. Stage 1 Results

In [ ]:
# Cell 11 — Stage 1 results
import json

with open('outputs/exp2/stage1/history.json') as f:
    history = json.load(f)

best = max(history, key=lambda x: x['val_mIoU'])
print(f"Stage 1 (COD10K pretrain):")
print(f"  Best epoch : {best['epoch']}")
print(f"  val mIoU   : {best['val_mIoU']:.4f}")
print(f"  val F1     : {best['val_F1']:.4f}")
print(f"  val MAE    : {best['val_MAE']:.4f}")

---
## 7. Stage 2 — ACD1K Fine-Tuning (Hyperparameter Sweep)

Stage 2 loads the best Stage-1 checkpoint and fine-tunes on ACD1K (748
military camouflage images) with a uniformly lower learning rate to adapt
to the military domain while minimising catastrophic forgetting of the
COD10K camouflage features.

We sweep over three learning rates: **1e-4, 1e-5, 5e-6** with 20 epochs
each to find the best LR before the final 50-epoch run.

In [ ]:
# Cell 12 — Stage 2 sweep run 1 (lr=1e-4)
!PYTHONPATH=/content/AI-final-project/src \
 python src/train_exp2.py \
    --stage 2 \
    --lr 1e-4 --epochs 20 --batch_size 16 --accum_steps 1 \
    --stage1_weights outputs/exp2/stage1/best_model.pth \
    --data_root data/ --splits_dir splits/ \
    --output_dir outputs/exp2/sweep_s2_lr1e4

In [ ]:
import json

with open('outputs/exp2/sweep_s2_lr1e4/history.json') as f:
    history = json.load(f)

best = max(history, key=lambda x: x['val_mIoU'])
print(f"Stage 2 sweep (lr=1e-4):")
print(f"  Best epoch : {best['epoch']}")
print(f"  val mIoU   : {best['val_mIoU']:.4f}")
print(f"  val F1     : {best['val_F1']:.4f}")
print(f"  val MAE    : {best['val_MAE']:.4f}")

In [ ]:
# Cell 14 — Stage 2 sweep run 2 (lr=1e-5)
!PYTHONPATH=/content/AI-final-project/src \
 python src/train_exp2.py \
    --stage 2 \
    --lr 1e-5 --epochs 20 --batch_size 16 --accum_steps 1 \
    --stage1_weights outputs/exp2/stage1/best_model.pth \
    --data_root data/ --splits_dir splits/ \
    --output_dir outputs/exp2/sweep_s2_lr1e5

In [ ]:
import json

with open('outputs/exp2/sweep_s2_lr1e5/history.json') as f:
    history = json.load(f)

best = max(history, key=lambda x: x['val_mIoU'])
print(f"Stage 2 sweep (lr=1e-5):")
print(f"  Best epoch : {best['epoch']}")
print(f"  val mIoU   : {best['val_mIoU']:.4f}")
print(f"  val F1     : {best['val_F1']:.4f}")
print(f"  val MAE    : {best['val_MAE']:.4f}")

In [ ]:
# Cell 16 — Stage 2 sweep run 3 (lr=5e-6)
!PYTHONPATH=/content/AI-final-project/src \
 python src/train_exp2.py \
    --stage 2 \
    --lr 5e-6 --epochs 20 --batch_size 16 --accum_steps 1 \
    --stage1_weights outputs/exp2/stage1/best_model.pth \
    --data_root data/ --splits_dir splits/ \
    --output_dir outputs/exp2/sweep_s2_lr5e6

In [ ]:
import json

with open('outputs/exp2/sweep_s2_lr5e6/history.json') as f:
    history = json.load(f)

best = max(history, key=lambda x: x['val_mIoU'])
print(f"Stage 2 sweep (lr=5e-6):")
print(f"  Best epoch : {best['epoch']}")
print(f"  val mIoU   : {best['val_mIoU']:.4f}")
print(f"  val F1     : {best['val_F1']:.4f}")
print(f"  val MAE    : {best['val_MAE']:.4f}")

---
## 8. Sweep Results — Pick Best LR

In [ ]:
# Cell 18 — Compare all Stage 2 sweep results
import json, os

runs = {
    'sweep_s2_lr1e4': 'outputs/exp2/sweep_s2_lr1e4/history.json',
    'sweep_s2_lr1e5': 'outputs/exp2/sweep_s2_lr1e5/history.json',
    'sweep_s2_lr5e6': 'outputs/exp2/sweep_s2_lr5e6/history.json',
}

print("=== Experiment 2 — Stage 2 Sweep Results ===")
for name, path in runs.items():
    if os.path.exists(path):
        with open(path) as f:
            h = json.load(f)
        best = max(h, key=lambda x: x['val_mIoU'])
        print(f"{name:<20} mIoU={best['val_mIoU']:.4f}  "
              f"F1={best['val_F1']:.4f}  "
              f"MAE={best['val_MAE']:.4f}  "
              f"@ ep{best['epoch']}")
    else:
        print(f"{name:<20} NOT FOUND")

---
## 9. Stage 2 — Final Training Run

Run the full 50-epoch training with the best learning rate from the sweep.
Adjust the `--lr` value below based on your sweep results.

In [ ]:
# Cell 20 — Stage 2 final training (50 epochs, best LR from sweep)
# ⚠️  UPDATE --lr below to the best LR from your sweep results
!PYTHONPATH=/content/AI-final-project/src \
 python src/train_exp2.py \
    --stage 2 \
    --lr 1e-5 --epochs 50 --batch_size 16 --accum_steps 1 \
    --stage1_weights outputs/exp2/stage1/best_model.pth \
    --data_root data/ --splits_dir splits/ \
    --output_dir outputs/exp2/final

---
## 10. Results & Export

In [ ]:
# Cell 21 — Final results
import json

with open('outputs/exp2/final/history.json') as f:
    h = json.load(f)
best = max(h, key=lambda x: x['val_mIoU'])
print(f"Experiment 2 — Final Stage 2 Results:")
print(f"  Best epoch : {best['epoch']}")
print(f"  val mIoU   : {best['val_mIoU']:.4f}")
print(f"  val F1     : {best['val_F1']:.4f}")
print(f"  val MAE    : {best['val_MAE']:.4f}")
print(f"  val loss   : {best['val_loss']:.4f}")

In [ ]:
# Cell 22 — Full Experiment 2 results summary
import json, os

runs = {
    'stage1':          'outputs/exp2/stage1/history.json',
    'sweep_s2_lr1e4':  'outputs/exp2/sweep_s2_lr1e4/history.json',
    'sweep_s2_lr1e5':  'outputs/exp2/sweep_s2_lr1e5/history.json',
    'sweep_s2_lr5e6':  'outputs/exp2/sweep_s2_lr5e6/history.json',
    'final':           'outputs/exp2/final/history.json',
}

print("=== Experiment 2 Full Results ===")
for name, path in runs.items():
    if os.path.exists(path):
        with open(path) as f:
            h = json.load(f)
        best = max(h, key=lambda x: x['val_mIoU'])
        print(f"{name:<20} mIoU={best['val_mIoU']:.4f}  "
              f"F1={best['val_F1']:.4f}  "
              f"MAE={best['val_MAE']:.4f}  "
              f"@ ep{best['epoch']}")
    else:
        print(f"{name:<20} NOT FOUND")

In [ ]:
# Cell 23 — Push results to GitHub and download checkpoint
!git config --global user.email "smansouri7@my.bcit.ca"
!git config --global user.name "sepehr-mansouri"

# Add train_exp2.py to repo if new
!git add src/train_exp2.py

# Add all experiment 2 outputs
!git add outputs/exp2/stage1/config.json \
         outputs/exp2/stage1/history.json \
         outputs/exp2/sweep_s2_lr1e4/history.json \
         outputs/exp2/sweep_s2_lr1e5/history.json \
         outputs/exp2/sweep_s2_lr5e6/history.json \
         outputs/exp2/final/history.json \
         outputs/exp2/final/config.json

!git commit -m "Exp2: Stage1 pretrain + Stage2 sweep + final (transfer learning)"
!git push

# Download final checkpoint
from google.colab import files
files.download('outputs/exp2/final/best_model.pth')
files.download('outputs/exp2/final/history.json')